# `explain`

Explainability: permutation importance (always) + SHAP (if installed).

> Part of the crop yield & recommendation project. Imported by `run_all.py` (via the notebook loader in `src/__init__.py`) or runnable directly here.

In [ ]:
# Setup: make the project root importable when this notebook is opened directly in Jupyter
import sys
from pathlib import Path
_root = Path.cwd().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src import config as C
from src.features import build_preprocessor
from src.train_recommend import FEATURES as RECO_FEATURES
from src.train_yield import CAT_COLS, load_data
from src.utils import have, savefig

warnings.filterwarnings("ignore")

In [ ]:
def _yield_explainer(num_cols):
    """Single tree model used for explanation (LightGBM > XGBoost > RandomForest)."""
    if have("lightgbm"):
        from lightgbm import LGBMRegressor
        est = LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=63, verbose=-1, random_state=C.SEED)
    elif have("xgboost"):
        from xgboost import XGBRegressor
        est = XGBRegressor(n_estimators=300, learning_rate=0.06, max_depth=8, verbosity=0, random_state=C.SEED)
    else:
        est = RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=C.SEED)
    return Pipeline([("prep", build_preprocessor("ordinal", num_cols, CAT_COLS)), ("model", est)])

In [ ]:
def explain_yield():
    df, features, num_cols, _ = load_data()
    tr = df["Year"] <= C.TRAIN_END_YEAR
    pipe = _yield_explainer(num_cols).fit(df.loc[tr, features], df.loc[tr, "y"])
    te = df[~tr].sample(min(3000, int((~tr).sum())), random_state=C.SEED)
    Xs, ys = te[features], te["y"]

    pi = permutation_importance(pipe, Xs, ys, scoring="r2", n_repeats=5, random_state=C.SEED)
    imp = pd.Series(pi.importances_mean, index=features).sort_values()
    imp.to_csv(C.REPORTS / "yield_permutation_importance.csv", header=["importance"])
    plt.figure(figsize=(7, 4)); imp.plot.barh(color="seagreen")
    plt.xlabel("Drop in R2 when feature is shuffled"); plt.title("Yield model - permutation importance")
    savefig("yield_permutation_importance.png")

    # per-crop importance heatmap
    top = df["Crop"].value_counts().head(8).index
    rows = {}
    for crop in top:
        sub = te[te["Crop"] == crop]
        if len(sub) < 60:
            continue
        r = permutation_importance(pipe, sub[features], sub["y"], scoring="r2", n_repeats=3, random_state=C.SEED)
        rows[crop] = pd.Series(r.importances_mean, index=features)
    if rows:
        hm = pd.DataFrame(rows).T.drop(columns=["Crop"], errors="ignore")
        plt.figure(figsize=(8, 4.5)); sns.heatmap(hm, annot=True, fmt=".2f", cmap="YlGn")
        plt.title("Per-crop feature importance (permutation, R2 drop)")
        savefig("yield_per_crop_importance.png")

    if have("shap"):
        import shap
        Xt = pipe["prep"].transform(Xs)
        names = num_cols + CAT_COLS
        sv = shap.TreeExplainer(pipe["model"]).shap_values(Xt)
        shap.summary_plot(sv, Xt, feature_names=names, show=False)
        savefig("yield_shap_summary.png")
        shap.summary_plot(sv, Xt, feature_names=names, plot_type="bar", show=False)
        savefig("yield_shap_bar.png")
    else:
        print("  (shap not installed - skipped SHAP plots; permutation importance done)")

In [ ]:
def explain_reco():
    df = pd.read_csv(C.RECO_CLEAN)
    X, y = df[RECO_FEATURES], df["crop"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=C.SEED)
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=C.SEED).fit(Xtr, ytr)
    pi = permutation_importance(rf, Xte, yte, scoring="accuracy", n_repeats=10, random_state=C.SEED)
    imp = pd.Series(pi.importances_mean, index=RECO_FEATURES).sort_values()
    imp.to_csv(C.REPORTS / "reco_permutation_importance.csv", header=["importance"])
    plt.figure(figsize=(7, 4)); imp.plot.barh(color="darkorange")
    plt.xlabel("Drop in accuracy when feature is shuffled"); plt.title("Recommendation - permutation importance")
    savefig("reco_permutation_importance.png")

    if have("shap"):
        import shap
        try:
            sv = shap.TreeExplainer(rf).shap_values(Xte)
            shap.summary_plot(sv, Xte, plot_type="bar", show=False)
            savefig("reco_shap_bar.png")
        except Exception as e:  # shap multiclass output format differs between versions
            print(f"  SHAP (recommendation) skipped: {e}")

In [ ]:
def main():
    print("[4/4] Explainability")
    explain_yield()
    explain_reco()

In [ ]:
if __name__ == "__main__":
    main()